# Evaluation of model on test set 🎯📈

 This script is for evaluating a model trained using agile modeling on a test set (manually annotated) saved as a csv file (see create_test_set_data.ipynb for more info)

## First load all the necessary stuff

In [ ]:
import collections
from etils import epath
import ipywidgets as widgets
from IPython.display import display as ipy_display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.io import wavfile
import shutil
import tensorflow as tf
import tqdm
import os
import json

from chirp import audio_utils
from chirp.inference import embed_lib
from chirp.inference import tf_examples
from chirp.inference.search import bootstrap
from chirp.inference.search import search
from chirp.inference.search import display
from chirp.inference.classify import classify
from chirp.inference.classify import data_lib

In [ ]:
# Check GPU is used, return [] if 
tf.config.list_physical_devices('GPU')

In [ ]:
path_to_config = './config_dict.json'
paths_dictionary = json.load(open(path_to_config, 'r'))

# Define paths frpm the config dictionary
sample_data_folder = os.path.join(paths_dictionary['working_repo'], paths_dictionary['sample_data_folder'])
output_directory = os.path.join(paths_dictionary['working_repo'], paths_dictionary['output_directory'])
dataset_folder = paths_dictionary['deployment_folder']
model_folder = os.path.join(paths_dictionary['working_repo'], paths_dictionary['embedding_models_folder'])
model_name = paths_dictionary['model_name']

unlabeled_audio_pattern = os.path.join(sample_data_folder, dataset_folder, 'raw_audio/*.[wW][aA][vV]')

print("Path to dataset: ", sample_data_folder)
print("Output data folder: ", output_directory)
print(f"Working on dataset: {dataset_folder} \nUsing {model_name}")

In [ ]:
from utils_agile_model import choose_embedding_model

embed_fn, config = choose_embedding_model(model_name)

# For readability later in the code
sample_rate = config.embed_fn_config.model_config.sample_rate
hop_size_s = config.embed_fn_config.model_config.hop_size_s
window_size_s = config.embed_fn_config.model_config.window_size_s

print(f"Ready to use Agile Modeling with '{model_name}' model")
print(f"Sampling rate:{sample_rate}Hz, Window size:{window_size_s}sec, Hop size:{hop_size_s}sec")


# Specify a glob pattern matching any number of wave files.
# Use [wW][aA][vV] to match .wav or .WAV files
unlabeled_audio_pattern = os.path.join(sample_data_folder, dataset_folder, 'raw_audio/*.[wW][aA][vV]')

print("Working on dataset: ", dataset_folder)
print("Input data folder: ", sample_data_folder)
print("Output data folder: ", output_directory)

## Load the model and the test set

In [ ]:
# Load the model to be used for evaluation
# # check load the model
model_path = os.path.join(output_directory, dataset_folder, model_name, 'HT2026_model_110626.keras')
model = tf.keras.models.load_model(model_path)
print("Model loaded from: ", model_path)

In [ ]:
# Load the test set
testset_path = "/home/reindert/Valentin_REVO/WaddenSea_KIM/Kim_Heligo Data/Outputs/winter 2024/surfperch/test_set/"
pickle_filename = 'test_set.pkl'
testset_df = pd.read_pickle(os.path.join(testset_path, pickle_filename))
testset_df['Label'].value_counts()
# testset_df.head()

In [ ]:
# If label is only nan, set to [] insteadt, if there is a a nan in list of labels, remove the nan
def safe_to_list(x):
    if isinstance(x, list):
        return [item for item in x if isinstance(item, str)]
    if isinstance(x, float):
        return []
    if isinstance(x, str):
        try:
            import ast
            parsed = ast.literal_eval(x)
            return [item for item in parsed if isinstance(item, str)] if isinstance(parsed, list) else [x]
        except:
            return [x]
    return []

testset_df['Label'] = testset_df['Label'].apply(safe_to_list)

In [ ]:
def discard_labels(df, labels_to_discard: list, label_col: str = "Label"):
    """
    Remove specified labels from each row's label list in the testset.
    Rows that end up with no labels after filtering are dropped.

    Parameters
    ----------
    df : pd.DataFrame
        The testset dataframe with a column containing lists of labels.
    labels_to_discard : list
        Labels to remove (e.g. ["Artificial_Knock", "Seal_Squaka"]).
    label_col : str
        Name of the column containing label lists. Defaults to "Label".

    Returns
    -------
    pd.DataFrame
        A copy of the dataframe with the specified labels removed.
        Rows where all labels were discarded are dropped.
    """
    discard_set = set(labels_to_discard)

    df_filtered = df.copy()
    df_filtered[label_col] = df_filtered[label_col].apply(
        lambda labels: [l for l in labels if l not in discard_set]
    )

    n_empty = (df_filtered[label_col].apply(len) == 0).sum()
    if n_empty > 0:
        print(f"{n_empty} row(s) have all labels discarded and will be treated as noise.")

    return df_filtered.reset_index(drop=True)

In [ ]:
# Sanity check to see some of the weird labels:

# Return all rows where Label contains 'Seal_Grunt' in the label list
label_to_look_for = 'Artificial_Mooring '

filename = testset_df[testset_df["Label"].apply(lambda labels: label_to_look_for in labels)]['filename']
print(f"There is {len(filename)} rows with \"{label_to_look_for}\"")

for i in range(len(filename)):
    print(filename.iloc[i])


In [ ]:
# Display all classes in the testset
class_names = list(set([item for sublist in testset_df["Label"].tolist() for item in sublist]))
print(class_names)

In [ ]:
# Write a list of the labels to discard:
labels_to_discard = ["Artificial_Mooring ", " Artificial_Mooring", "Artificial_Mooring", "Artificial_Knock",
                    'Seal_Burp', 'Seal_Clap', 'Seal_Growl', 'Seal_Growl ', 'Seal_Grunt',
                    'Seal_Huhu', 'Seal_Huhu ', 'Seal_Moan', 'Seal_Moan_HF', 'Seal_Rup',
                    'Seal_Rup_LF', 'Seal_Rup_LF ', 'Seal_Two_Part', 'Seal_mev']


# Filter the test set for classes that are in the model
testset_df_filtered = discard_labels(testset_df, labels_to_discard)

testset_df_filtered['Label'].value_counts()

# Map the label from the testset and the model
label_mapping = {
    "Seal_Squak":       "squak",      # model class name
    "Seal_Squaka":     "squak",
    "Seal_Squak_Moan": "squak",
    "Seal_Click_Series": "grunt",
    # anything not covered → None or "unknown"
}

def map_labels(label_list, mapping):
    mapped = {mapping[l] for l in label_list if l in mapping}
    return list(mapped)  # may be empty if all labels are unmapped

testset_df_filtered["mapped_labels"] = testset_df_filtered["Label"].apply(lambda l: map_labels(l, label_mapping))

In [ ]:
import numpy as np
import tensorflow as tf

model_class_names = ["squak", "grunt", "noise", "absent_class"]  # order must match logit positions
eval_class_names = ["squak", "grunt", "noise", "class4"]  # drop the absent class from evaluation

def predict_multilabel(model, emb, class_names, threshold=0.5, ignore=["absent_class"]):
    logits = model.predict(emb, verbose=0)
    probs = tf.nn.sigmoid(logits).numpy()[0]
    predicted = [class_names[i] for i, p in enumerate(probs)
                 if p >= threshold and class_names[i] not in ignore]
    return predicted

quick_test = testset_df_filtered.iloc[:1000]

y_pred = [predict_multilabel(model, emb, model_class_names) for emb in quick_test["Embedding"]]

# For ground truth: empty mapped_labels → noise
def get_true_labels(mapped):
    return mapped if len(mapped) > 0 else ["noise"]

y_true = quick_test["mapped_labels"].apply(get_true_labels).tolist()

# Evaluate
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report, multilabel_confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

mlb = MultiLabelBinarizer(classes=eval_class_names)
y_true_bin = mlb.fit_transform(y_true)
y_pred_bin = mlb.transform(y_pred)

print(classification_report(y_true_bin, y_pred_bin, target_names=eval_class_names))

fig, axes = plt.subplots(1, len(eval_class_names), figsize=(4 * len(eval_class_names), 4))
for i, (cls, ax) in enumerate(zip(eval_class_names, axes)):
    ConfusionMatrixDisplay(multilabel_confusion_matrix(y_true_bin, y_pred_bin)[i],
                           display_labels=["not " + cls, cls]).plot(ax=ax)
    ax.set_title(cls)
plt.tight_layout()
plt.show()